In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [7]:
from src.data_loader import load_raw_data

df = load_raw_data()

In [8]:
from src.preprocessing import clean_data

df = clean_data(df)
print(df.shape)

(1337, 7)


In [13]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["charges"])
y = df["charges"]

Feature Engineering

In [ ]:
X = X.copy()

X["bmi_obese"] = (X["bmi"] >= 30).astype(int)
X[["bmi", "bmi_obese"]].head(10)

,bmi,bmi_obese
0,27.900,0
1,33.770,1
2,33.000,1
3,22.705,0
4,28.880,0
5,25.740,0
6,33.440,1
7,27.740,0
8,29.830,0
9,25.840,0


In [15]:
X["smoker_obese"] = (
    (X["smoker"] == "yes") & (X["bmi"] >= 30)
).astype(int)

In [16]:
X[["bmi", "smoker", "bmi_obese", "smoker_obese"]].head(10)

,bmi,smoker,bmi_obese,smoker_obese
0,27.900,yes,0,0
1,33.770,no,1,0
2,33.000,no,1,0
3,22.705,no,0,0
4,28.880,no,0,0
5,25.740,no,0,0
6,33.440,no,1,0
7,27.740,no,0,0
8,29.830,no,0,0
9,25.840,no,0,0


In [17]:
print(X["bmi_obese"].value_counts())
print()
print(X["smoker_obese"].value_counts())

bmi_obese
1    706
0    631
Name: count, dtype: int64

smoker_obese
0    1192
1     145
Name: count, dtype: int64


In [18]:
X["age_sq"] = X["age"] ** 2

X[["age", "age_sq"]].head()

,age,age_sq
0,19,361
1,18,324
2,28,784
3,33,1089
4,32,1024


In [19]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [21]:
binary_col=["bmi_obese","smoker_obese"]
numerical_features = [
    col for col in X.select_dtypes(include="number").columns
    if col not in binary_col
]

categorical_col=X.select_dtypes(exclude="number").columns.tolist()

In [22]:
numerical_features

['age', 'bmi', 'children', 'age_sq']

In [23]:
categorical_col

['sex', 'smoker', 'region']

In [26]:
X_train, X_test, y_train, y_test=  train_test_split(
          X,
        y,
    test_size=0.2,
    random_state=42,
    stratify=X["smoker"]
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(1069, 9)
(268, 9)
(1069,)
(268,)


In [29]:
print(X_train["smoker"].value_counts(normalize=True)* 100)
print(X_test["smoker"].value_counts(normalize=True)* 100)

smoker
no     79.513564
yes    20.486436
Name: proportion, dtype: float64
smoker
no     79.477612
yes    20.522388
Name: proportion, dtype: float64


In [30]:
X_train.isnull().sum()

age             0
sex             0
bmi             0
children        0
smoker          0
region          0
bmi_obese       0
smoker_obese    0
age_sq          0
dtype: int64

In [33]:
for col in categorical_col:
    print(col, ":", X_train[col].unique())
    print("------------")

sex : <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str
------------
smoker : <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str
------------
region : <ArrowStringArray>
['northwest', 'southeast', 'northeast', 'southwest']
Length: 4, dtype: str
------------


In [31]:
X_test.isnull().sum().sum()

np.int64(0)

In [ ]:
from sklearn.preprocessing import  OneHotEncoder
X_train_encoded=OneHotEncoder(

 drop="first",
    handle_unknown="ignore",
    sparse_output=False
)
X_train_encoded.fit_transform( X_train[categorical_col])


array([[0., 0., 1., 0., 0.],
       [1., 0., 0., 1., 0.],
       [1., 0., 0., 1., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.]], shape=(1069, 5))

In [38]:
print(X_train_encoded.get_feature_names_out(categorical_col))

['sex_male' 'smoker_yes' 'region_northwest' 'region_southeast'
 'region_southwest']


In [ ]:
v